(sorting)=

# Sorting

```{contents}
:local:
:depth: 2
```

Sorting produces an ordered permutation of the input. The algorithms here mutate an array in place. Their counted key comparisons exclude loop-bound checks; swaps and insertion shifts are reported separately. All methods use the same explicit comparer.

## Swap and Preservation

```{index} Swap and Preservation
```

A swap exchanges two positions without losing a value. A temporary variable or tuple assignment preserves both originals; assigning values[a]=values[b] and then values[b]=values[a] would duplicate one value. Swapping a position with itself is harmless but adds unnecessary work.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={10,20,30};
(values[0],values[2])=(values[2],values[0]);
Console.WriteLine(string.Join(", ",values));

The elementary sorts use Θ(1) auxiliary state beyond the input. A comparison policy must be consistent; compare integer keys with CompareTo rather than subtracting them, which can overflow. An in-place routine changes the supplied array, so clone it when preserving arrival order matters.

## Selection Sort and Its Prefix Invariant

```{index} Selection Sort and Its Prefix Invariant
```

At the start of pass s, [0,s) contains the smallest s input occurrences in sorted order. Scan [s,n), retain the smallest index, then swap it into s. The inner-loop invariant tracks the smallest value in the scanned suffix prefix.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={5,2,9,1,3};
var counts=ElementarySort.Selection(values,Comparer<int>.Default);
Console.WriteLine($"sorted={string.Join(",",values)}, comparisons={counts.Comparisons}, swaps={counts.Swaps}");
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


The next chosen minimum extends the sorted prefix and leaves its elements no larger than the remaining suffix. Swaps preserve occurrences. There are exactly (n−1)+(n−2)+…+1=n(n−1)/2 key comparisons and at most n−1 actual swaps. Best and worst time are Θ(n²) for growing n, including sorted input. The shown implementation skips self-swaps. A long counter prevents count overflow on moderate large n.

## Insertion Sort, Shifts, and Nearly Sorted Data

```{index} Insertion Sort, Shifts, and Nearly Sorted Data
```

Before pass i, [0,i) is sorted. Save the current item, shift strictly greater predecessors right, and insert the saved item in the gap. During shifting the saved value is outside the array; include it when reasoning about occurrence preservation.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={3,1,2};
for(int i=1;i<values.Length;i++)
{
 int current=values[i],j=i-1;
 Console.WriteLine($"pass={i}, saved={current}, before=[{string.Join(",",values)}]");
 while(j>=0 && values[j]>current)
 {
  values[j+1]=values[j];j--;
  Console.WriteLine($"shift; gap={j+1}, array=[{string.Join(",",values)}], saved={current}");
 }
 values[j+1]=current;
 Console.WriteLine($"insert at {j+1}: [{string.Join(",",values)}]");
}

Equal keys are not shifted, preserving their arrival order. The sorted best case takes n−1 key comparisons and zero shifts; reverse-distinct input takes n(n−1)/2 comparisons and shifts. More precisely, shifts equal the number of inversions: pairs i<j whose earlier key is greater. With I inversions, this insertion implementation takes Θ(n+I) work for n≥1, which explains nearly sorted behavior rather than merely calling it fast.

## Bubble Sort and a Settled Suffix

```{index} Bubble Sort and a Settled Suffix
```

Each left-to-right pass moves the largest remaining item to the current end. The suffix beyond end is already sorted and in final positions; shorten the next pass. If a pass makes no swaps, the remaining prefix has no adjacent inversions and is sorted.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int[] values in new[]{new[]{1,2,3,4},new[]{4,3,2,1},Array.Empty<int>()})
{
 var counts=ElementarySort.Bubble(values,Comparer<int>.Default);
 Console.WriteLine($"[{string.Join(",",values)}]: comparisons={counts.Comparisons}, swaps={counts.Swaps}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


Strictly out-of-order swaps make this bubble variant stable. The no-swap exit gives Θ(n) best-case time on sorted inputs; the shrinking-end reverse case has n(n−1)/2 comparisons and swaps, hence Θ(n²). Empty/singleton arrays need no passes. These claims apply to this variant, not to a fixed-pass or full-width bubble implementation.

## Stability Is Visible on Records

```{index} Stability Is Visible on Records
```

Selection sort’s long-distance swap can reverse equal keys even though the minimum scan uses a strict comparison. Insertion and bubble preserve ties in their shown forms. Observe IDs as well as amounts.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var original=new[]{("A",200),("B",200),("C",100)};
var comparer=Comparer<(string,int)>.Create((a,b)=>a.Item2.CompareTo(b.Item2));
foreach(string algorithm in new[]{"selection","insertion","bubble"})
{
 var copy=original.ToArray();
 if(algorithm=="selection")ElementarySort.Selection(copy,comparer);
 else if(algorithm=="insertion")ElementarySort.Insertion(copy,comparer);
 else ElementarySort.Bubble(copy,comparer);
 Console.WriteLine($"{algorithm}: {string.Join(",",copy.Select(x=>x.Item1))}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


Selecting C and swapping it with A gives C,B,A: the two 200-value invoices reverse. Stable output is C,A,B. Stability describes equal comparison keys, not equality of complete records. Do not infer stable behavior from one lucky run of an algorithm without that guarantee.

Library sorting, mutation, and stable report ordering are taught in {doc}`Chapter 15 <../15_arrays_linked_lists/1504_searching_sorting>`. Use those contracts when comparing these optional implementations.


Select by contract and workload: insertion can suit tiny or nearly sorted teaching inputs, but quadratic sorts scale poorly in the worst case. Chapter 21’s stable merge sort has Θ(n log n) time and Θ(n) auxiliary storage. The library algorithms are the usual starting point for application code; custom comparisons still require a consistent order. Mutable records are not deep-cloned by making an array copy.

## Exercise

```{index} Exercise
```

Run insertion sort on sorted and reverse arrays of length four. Report comparisons and shifts; relate shifts to inversions.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[][] cases={new[]{1,2,3,4},new[]{4,3,2,1}};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[][] cases={new[]{1,2,3,4},new[]{4,3,2,1}};
foreach(int[] values in cases)
{
 var counts=ElementarySort.Insertion(values,Comparer<int>.Default);
 Console.WriteLine($"comparisons={counts.Comparisons}, shifts={counts.Shifts}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


comparisons=3, shifts=0
comparisons=6, shifts=6


```{rubric} Footnotes
```
[^1]: [Microsoft’s Array.Sort documentation](https://learn.microsoft.com/en-us/dotnet/api/system.array.sort?view=net-10.0) specifies its unstable tie behavior.
[^2]: [Microsoft’s OrderBy documentation](https://learn.microsoft.com/en-us/dotnet/api/system.linq.enumerable.orderby?view=net-10.0) specifies stable ordering and deferred execution; ToArray materializes the report.